# Parent-Child Retrieval (Enterprise AI Pattern)

Parent-Child Retrieval is one of the **most important Advanced RAG techniques** and is frequently asked in **EPAM, Microsoft, Amazon, Deloitte, TCS, Accenture, Cognizant**, and other enterprise AI interviews.

Typical questions include:

- What is Parent-Child Retrieval?
- Why not use normal chunking?
- Parent Chunk vs Child Chunk?
- How does Parent-Child Retrieval improve RAG?
- Explain the architecture.

---

# 1. What is Parent-Child Retrieval?

## Definition

Parent-Child Retrieval is a retrieval strategy where:

- **Small child chunks** are stored in the vector database for **accurate semantic search**.
- After retrieval, the **larger parent chunk** is returned to the LLM to provide **complete context**.

This combines the precision of small chunks with the context of larger chunks.

---

## Interview Answer

> Parent-Child Retrieval stores small child chunks in the vector database for accurate semantic retrieval, but instead of sending those small chunks directly to the LLM, it retrieves the corresponding larger parent document section. This preserves context while maintaining high retrieval accuracy.

---

# 2. Why Do We Need Parent-Child Retrieval?

Suppose we have a 10-page HR policy.

Traditional chunking:

```text
Chunk 1 (500 tokens)

Chunk 2 (500 tokens)

Chunk 3 (500 tokens)

Chunk 4 (500 tokens)
```

User asks

```text
Can unused leave be carried forward?
```

The answer may be split across two chunks.

Example

Chunk 2

```text
Unused leave...
```

Chunk 3

```text
...can be carried forward for 5 days.
```

The LLM receives only Chunk 2.

Result

❌ Incomplete answer.

---

# 3. Parent-Child Idea

Instead of storing only one chunk:

Large Parent

```text
HR Policy

↓

2000 Tokens
```

↓

Split into

```text
Child 1 (300)

Child 2 (300)

Child 3 (300)

Child 4 (300)

Child 5 (300)
```

---

Store

```text
Vector DB

↓

Child Chunks
```

Retrieve

```text
Child 3
```

Then return

```text
Entire Parent Chunk
```

Now Claude receives enough context.

---

# 4. Architecture

```text
                 PDF

                  │

                  ▼

          Parent Splitter

                  │

          Parent Chunks

                  │

                  ▼

          Child Splitter

                  │

          Child Chunks

                  │

                  ▼

             Embeddings

                  │

                  ▼

       OpenSearch / Qdrant

                  │

                  ▼

          Child Retrieved

                  │

                  ▼

     Parent Document Lookup

                  │

                  ▼

      AWS Bedrock / Azure OpenAI

                  │

                  ▼

             Final Answer
```

---

# AWS + Azure Components

| Layer | AWS | Azure |
|--------|------|--------|
| Storage | S3 | Blob Storage |
| Vector DB | OpenSearch/Qdrant | Azure AI Search/Qdrant |
| LLM | Bedrock | Azure OpenAI |
| Embeddings | Titan | text-embedding-3-large |

---

# 5. Ingestion Flow

```text
PDF

↓

Parent Split

↓

Child Split

↓

Embeddings

↓

Vector Database

↓

Parent Store
```

Notice

Parent

↓

Document Store

Child

↓

Vector DB

---

# 6. Query Flow

```text
Question

↓

Embedding

↓

Vector Search

↓

Best Child Chunk

↓

Find Parent Chunk

↓

LLM

↓

Answer
```

---

# 7. Example

Parent Chunk

```text
Annual Leave Policy

Employees receive 20 annual leave days.

Unused leave can be carried forward for 5 days.

Manager approval is mandatory.

Medical leave is separate.
```

---

Child Chunks

Child 1

```text
Employees receive 20 annual leave days.
```

---

Child 2

```text
Unused leave can be carried forward...
```

---

Child 3

```text
Manager approval is mandatory.
```

---

Question

```text
Can unused leave be carried forward?
```

Retriever

↓

Child 2

↓

Parent Lookup

↓

Entire Leave Policy

↓

LLM

---

# 8. LangChain Example

```python
# ==========================================================
# STEP 1 : Load Documents
# ==========================================================

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("leave_policy.pdf")

docs = loader.load()


# ==========================================================
# STEP 2 : Create Parent Splitter
#
# Parent chunks are larger and preserve context.
# ==========================================================

from langchain.text_splitter import RecursiveCharacterTextSplitter

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

parent_docs = parent_splitter.split_documents(docs)


# ==========================================================
# STEP 3 : Create Child Splitter
#
# Child chunks are smaller for accurate retrieval.
# ==========================================================

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)


# ==========================================================
# STEP 4 : Create Parent Document Retriever
#
# Child chunks are embedded and indexed.
# Parent documents are stored separately.
# ==========================================================

from langchain.storage import InMemoryStore
from langchain.retrievers import ParentDocumentRetriever

store = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)


# ==========================================================
# STEP 5 : Add Documents
#
# Parent documents go to the document store.
# Child chunks go to the vector database.
# ==========================================================

retriever.add_documents(parent_docs)


# ==========================================================
# STEP 6 : Retrieve
#
# Search retrieves child chunks, but returns
# the corresponding parent document.
# ==========================================================

results = retriever.invoke(
    "What is the leave carry forward policy?"
)

for doc in results:
    print(doc.page_content)
```

> **Production Note:** In production, the `vector_store` would typically be backed by **Amazon OpenSearch**, **Qdrant**, or **Azure AI Search**, while the parent documents may be stored in an external document store (e.g., S3/Blob + metadata store) rather than `InMemoryStore`.

---

# 9. Production Architecture

```text
User

↓

FastAPI

↓

LangGraph

↓

Retriever

↓

Child Search

↓

OpenSearch

↓

Parent Lookup

↓

Reranker

↓

Bedrock

↓

Redis

↓

Response
```

---

# 10. Advantages

✅ Better retrieval accuracy

✅ Complete context

✅ Fewer hallucinations

✅ Better long-document support

---

# 11. Disadvantages

❌ More storage

❌ More indexing

❌ Additional lookup step

❌ Slightly higher latency

---

# 12. Best Practices

✅ Parent = 1500–2500 tokens

✅ Child = 300–500 tokens

✅ Metadata links between parent and child

✅ Combine with reranking

✅ Use semantic chunking where possible

---

# 13. Common Mistakes

❌ Parent too large

❌ Child too small

❌ No parent metadata

❌ Returning child chunks directly

---

# 14. Parent-Child vs Traditional RAG

| Traditional RAG | Parent-Child Retrieval |
|-----------------|------------------------|
| Stores one chunk | Stores parent + child |
| Retrieves chunk | Retrieves parent |
| May lose context | Preserves context |
| Simpler | More accurate |

---

# 15. Real Enterprise Example

### Healthcare Assistant

Medical guideline

```text
50 Pages
```

Question

```text
What are the contraindications for Drug X?
```

Retriever

↓

Small child section

↓

Returns

Entire guideline section

↓

LLM

↓

Accurate answer.

---

### HR Assistant

Question

```text
Can unused leave be carried forward?
```

Retriever

↓

Child chunk

↓

Parent HR policy

↓

Claude

↓

Answer

---

# 16. Interview Questions

### Q1. Why Parent-Child Retrieval?

To achieve accurate retrieval with small chunks while providing the LLM with richer context from larger parent sections.

---

### Q2. Why not use only large chunks?

Large chunks reduce retrieval precision because embeddings represent too much information.

---

### Q3. Why not use only small chunks?

Small chunks improve retrieval but often lose surrounding context, leading to incomplete answers.

---

### Q4. Where is Parent-Child Retrieval useful?

- HR policies
- Healthcare guidelines
- Legal contracts
- Financial reports
- Technical manuals

---

### Q5. Does Parent-Child Retrieval replace chunking?

No.

It is an advanced chunking strategy that uses **two levels of chunking**.

---

# 17. Traditional vs Parent-Child

Traditional

```text
Question

↓

Retriever

↓

Chunk

↓

LLM
```

Parent-Child

```text
Question

↓

Retriever

↓

Child

↓

Parent

↓

LLM
```

---

# 18. EPAM Senior Answer (3 Minutes)

> "Parent-Child Retrieval is an advanced RAG technique that improves both retrieval accuracy and contextual completeness. During ingestion, documents are first split into large parent chunks and then into smaller child chunks. Only the child chunks are embedded and indexed in a vector database such as Amazon OpenSearch, Qdrant, or Azure AI Search. When a user submits a query, semantic search identifies the most relevant child chunk, but instead of sending that fragment directly to the LLM, the retriever returns the associated parent chunk, which contains richer surrounding context. This allows Amazon Bedrock or Azure OpenAI to generate more accurate responses while reducing hallucinations and context fragmentation. Parent-Child Retrieval is especially valuable for long documents such as HR policies, healthcare guidelines, legal contracts, and technical manuals, where important information often spans multiple sections."